# Stage 2: DPO Training on LoRA-tuned Mistral 7B

**Environment:** Kaggle GPU (P100/T4)  
**Estimated time:** ~1-2 hours  
**Input:** SFT adapter from Stage 1 + preference dataset  
**Output:** DPO-aligned LoRA adapter in `/kaggle/working/dpo_adapter/`

## 1. Install Dependencies

In [1]:
!pip install -q trl peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.9 MB/s eta 0:00:00


   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/60.7 MB 194.4 MB/s eta 0:00:01

   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/60.7 MB 231.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 30.2/60.7 MB 230.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━ 38.1/60.7 MB 230.2 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00


## 2. Verify Data Paths
Run this first to find exact paths on Kaggle.

In [2]:
import os
print("=== All files in /kaggle/input/ ===")
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(root, f))

=== All files in /kaggle/input/ ===
/kaggle/input/datasets/aadityae/sft-adapter/adapter_model.safetensors
/kaggle/input/datasets/aadityae/sft-adapter/adapter_config.json
/kaggle/input/datasets/aadityae/sft-adapter/README.md
/kaggle/input/datasets/aadityae/sft-adapter/tokenizer.json
/kaggle/input/datasets/aadityae/sft-adapter/tokenizer_config.json
/kaggle/input/datasets/aadityae/sft-adapter/chat_template.jinja
/kaggle/input/datasets/aadityae/hr-preference-data/train.json
/kaggle/input/datasets/aadityae/hr-preference-data/test.json


## 3. Imports & Configuration
**IMPORTANT:** Update `SFT_ADAPTER_DIR` and `DATASET_PATH` below based on the output from the cell above.

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import DPOTrainer

# ── Hyperparameters ──
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# Quantization — float16 NOT bfloat16 (P100 doesn't support bf16)
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.float16
BNB_4BIT_USE_DOUBLE_QUANT = True

# LoRA (same config as SFT)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# DPO-specific
DPO_BETA = 0.1
DPO_EPOCHS = 2
DPO_BATCH_SIZE = 4
DPO_GRADIENT_ACCUMULATION_STEPS = 4
DPO_LEARNING_RATE = 5e-7
DPO_SAVE_STEPS = 500
DPO_LOGGING_STEPS = 25

# ══════════════════════════════════════════════════
# UPDATE THESE PATHS based on cell 2 output above!
# ══════════════════════════════════════════════════
SFT_ADAPTER_DIR = "/kaggle/input/datasets/aadityae/sft-adapter"  # <-- update if different
DATASET_PATH = "/kaggle/input/datasets/aadityae/hr-preference-data/train.json"  # <-- update if different

DPO_OUTPUT_DIR = "./dpo_output"
DPO_ADAPTER_DIR = "/kaggle/working/dpo_adapter"

# Resume training (set to checkpoint path or None)
RESUME_FROM_CHECKPOINT = None

# Disable W&B to avoid team permission issues
os.environ["WANDB_DISABLED"] = "true"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Memory: 17.1 GB


## 4. Load Preference Dataset
DPO requires three columns: `prompt`, `chosen`, `rejected`.

In [4]:
raw_ds = load_dataset("json", data_files=DATASET_PATH, split="train")

# Keep only the columns DPO needs
keep_cols = ["prompt", "chosen", "rejected"]
remove_cols = [c for c in raw_ds.column_names if c not in keep_cols]
if remove_cols:
    raw_ds = raw_ds.remove_columns(remove_cols)

print(f"Loaded {len(raw_ds)} preference pairs")
print(f"Columns: {raw_ds.column_names}")
print(f"\nSample:")
print(f"  Prompt:   {raw_ds[0]['prompt'][:100]}...")
print(f"  Chosen:   {raw_ds[0]['chosen'][:100]}...")
print(f"  Rejected: {raw_ds[0]['rejected'][:100]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 648 preference pairs
Columns: ['prompt', 'chosen', 'rejected']

Sample:
  Prompt:   What expenses are reimbursable during business travel?...
  Chosen:   During business travel, the following expenses are reimbursable:

- **Meals**: 
  - Company-provided...
  Rejected: During business travel, the following expenses are reimbursable:

1. Meals while traveling on Compan...


## 5. Load Base Model + SFT Adapter

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,  # torch.float16 directly
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare for kbit training
base_model = prepare_model_for_kbit_training(base_model)

# Load SFT adapter on top
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR, is_trainable=True)

print(f"Base model loaded: {BASE_MODEL}")
print(f"SFT adapter loaded from: {SFT_ADAPTER_DIR}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")
model.print_trainable_parameters()

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Base model loaded: mistralai/Mistral-7B-Instruct-v0.2
SFT adapter loaded from: /kaggle/input/datasets/aadityae/sft-adapter
Memory footprint: 4.59 GB
trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## 6. Configure DPO Training

In [6]:
from trl import DPOConfig

In [7]:
# training_args = DPOConfig(
#     output_dir=DPO_OUTPUT_DIR,
#     num_train_epochs=DPO_EPOCHS,
#     per_device_train_batch_size=DPO_BATCH_SIZE,
#     gradient_accumulation_steps=DPO_GRADIENT_ACCUMULATION_STEPS,
#     learning_rate=DPO_LEARNING_RATE,
#     warmup_steps=50,
#     lr_scheduler_type="cosine",
#     weight_decay=0.01,
#     fp16=False,   # DISABLED — P100 has issues with fp16 grad scaler + 4bit
#     bf16=False,   # DISABLED — P100 doesn't support bf16
#     gradient_checkpointing=True,
#     logging_steps=DPO_LOGGING_STEPS,
#     save_steps=DPO_SAVE_STEPS,
#     save_total_limit=3,
#     report_to="none",  # W&B disabled
#     run_name="dpo-lora-mistral7b",
#     optim="paged_adamw_8bit",
#     remove_unused_columns=False,
#     beta = DPO_BETA
# )

training_args = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=1,          # 1 epoch instead of 2
    per_device_train_batch_size=DPO_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRADIENT_ACCUMULATION_STEPS,
    learning_rate=1e-7,          # 5x lower
    warmup_steps=50,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=DPO_LOGGING_STEPS,
    save_steps=DPO_SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    run_name="dpo-lora-mistral7b",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    beta=0.5,                    # higher = stays closer to SFT model
)
# training_args.beta = DPO_BETA  # add this line after

print(f"DPO beta: {DPO_BETA}")
print(f"Learning rate: {DPO_LEARNING_RATE}")
print(f"Effective batch size: {DPO_BATCH_SIZE * DPO_GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {DPO_EPOCHS}")

DPO beta: 0.1
Learning rate: 5e-07
Effective batch size: 16
Epochs: 2


## 7. Initialize DPO Trainer
**Note:** Newer TRL versions changed parameter names. This cell handles both old and new API.

In [8]:
import trl
print(f"TRL version: {trl.__version__}")

# Try new API first, fall back to old API
try:
    dpo_trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=raw_ds,
        processing_class=tokenizer,
    )
    print("DPOTrainer initialized (new API)")
except TypeError as e:
    print(f"New API failed: {e}")
    print("Trying old API...")
    dpo_trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=raw_ds,
        tokenizer=tokenizer,
    )
    print("DPOTrainer initialized (old API)")

print(f"Training samples: {len(raw_ds)}")
print(f"Checkpoints every {DPO_SAVE_STEPS} steps")

TRL version: 0.29.0


Adding EOS to train dataset:   0%|          | 0/648 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/648 [00:00<?, ? examples/s]

DPOTrainer initialized (new API)
Training samples: 648
Checkpoints every 500 steps


## 8. Train

In [9]:
dpo_trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


## 9. Save DPO Adapter

In [ ]:
os.makedirs(DPO_ADAPTER_DIR, exist_ok=True)

dpo_trainer.model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

print(f"DPO adapter saved to {DPO_ADAPTER_DIR}")

for f in os.listdir(DPO_ADAPTER_DIR):
    size = os.path.getsize(os.path.join(DPO_ADAPTER_DIR, f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## 10. Zip for Download

In [ ]:
!zip -r /kaggle/working/dpo_adapter.zip /kaggle/working/dpo_adapter/
print("\nZipped! Save Version -> Output tab to download.")

## 11. Quick Sanity Check

In [ ]:
test_prompt = "[INST] What is the company's policy on remote work? [/INST]"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
# Load base + SFT adapter only (no DPO)
from peft import PeftModel

sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR, is_trainable=False)

test_prompt = "[INST] What is the company's policy on remote work? [/INST]"
inputs = tokenizer(test_prompt, return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)